# aDDM Tutorial

This notebook showcases the implementation of a modern aDDM, compatible with PyDDM.

### Load the data

In [1]:
from ast import literal_eval
import pandas as pd

# 1. Load data
df_raw = pd.read_csv('1ms_trial_data.csv')

# 2. Drop nuisance trials
to_drop = pd.read_csv("dropped_trials.csv").rename(columns={"parcode": "sub_id"})

df = df_raw.loc[
    ~df_raw.set_index(["sub_id", "trial"]).index.isin(
        to_drop.set_index(["sub_id", "trial"]).index
    )
    & (~df_raw["hidden"])
]

# 3. Adjustments
df['RT'] = (df['RT']*1000).astype(int) # RT unit scaling
df['fixation'] = df['fixation'].apply(literal_eval) # String to list as a result of csv saving
df['choice'] = df['choice'].replace({"left": 0, "right": 1}) # Map choice to 0 or 1

/var/folders/59/03h51wmn6xn1jhvb7kj9fjc40000gn/T/ipykernel_24878/4055317967.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['RT'] = (df['RT']*1000).astype(int) # RT unit scaling
/var/folders/59/03h51wmn6xn1jhvb7kj9fjc40000gn/T/ipykernel_24878/4055317967.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['fixation'] = df['fixation'].apply(literal_eval) # String to list as a result of csv saving
/var/folders/59/03h51wmn6xn1jhvb7kj9fjc40000gn/T/ipykernel_24878/4055317967.py:20: FutureWarning: Down

### Simulating data from empiricals

In [2]:
from simulation import get_corrected_empirical_distributions
import numpy as np

# Make empirical distributions
# value_diffs = np.arange(-4, 4.25, 0.25)
value_diffs = np.unique(df['avgWTP_left'] - df['avgWTP_right'])
legend = {
    "left": {1},
    "right": {2},
    "transition": {0}, 
    "blank_fixation": {4}
}
fixation_col = 'fixation'
left_value_col = 'avgWTP_left'
right_value_col = 'avgWTP_right'

empirical_distributions = get_corrected_empirical_distributions(
    df,
    value_diffs=value_diffs,
    legend=legend,
    fixation_col=fixation_col,
    left_value_col=left_value_col,
    right_value_col=right_value_col,
    cutoff=0.9
)

In [3]:
from simulation import generate_fixations

# Create sample trial conditions
dt = 0.01
seed = 42

trials = df.loc[
    (df['sub_id'] == 304) & (df['trial'] % 2 == 1),
    ['avgWTP_left', 'avgWTP_right']
].copy()
trials['fixation'] = None

rng = np.random.default_rng(seed)
trials_dict = []
for idx, r in trials.iterrows():
    fx = generate_fixations(
        dt, 
        r.avgWTP_left - r.avgWTP_right, 
        empirical_distributions,
        rng=rng
    )
    if fx is not None:
        trials_dict.append({
            "avgWTP_left": r.avgWTP_left,
            "avgWTP_right": r.avgWTP_right,
            "fixation": fx
        })

In [4]:
from simulation import simulate
import pyddm

model_conditions = {'drift_rate': 0.8, 'theta': 0.5, 'noise': 0.6}

results_df = simulate(dt, model_conditions, trials_dict, seed=seed, save_results=False)
# results_df['sub_id'] = f'seed{seed}_subjects{size}_sim'
# results_df['trial'] = range(1, len(trials_clean) + 1)
# results_df = results_df.rename(columns={'fixation': 'fix_sequence'})
results_df = results_df.drop(columns = ['trajectory'])

sample = pyddm.Sample.from_pandas_dataframe(
    results_df,
    choice_column_name="choice",
    rt_column_name="RT",
    choice_names=("left", "right")
)

print(f'Average RT: {results_df["RT"].mean():.2f} seconds (out of {len(results_df)} trials)')
results_df.head()

Average RT: 1.60 seconds (out of 100 trials)


,avgWTP_left,avgWTP_right,fixation,RT,choice
0,1.00,1.00,"(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",3.14,1
1,4.25,3.00,"(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",2.24,0
2,3.25,3.75,"(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, ...",3.73,1
3,3.00,2.75,"(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",5.63,1
4,1.00,4.00,"(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1.84,1


The above is the first half of the tutorial. Following is native parameter recovery by differential evolution.

In [5]:
# Define the model
def drift_function(avgWTP_left, avgWTP_right, fixation, d, x, t):
        fixation_index = min(int(t/dt), len(fixation)-1)
        current_fixation = fixation[fixation_index]
        if current_fixation == 0: # saccade
            drift_val = 0
        elif current_fixation == 1: # left
            drift_val = d * (avgWTP_left - avgWTP_right * model_conditions['theta'])
        else: # right
            drift_val = d * (avgWTP_left * model_conditions['theta'] - avgWTP_right)
        
        return np.ones_like(x) * drift_val
    
def noise_function(n, x, t):
    return np.ones_like(x) * n

model = pyddm.gddm(
    drift=drift_function,
    noise=noise_function,
    bound=1,
    nondecision=0,
    parameters={'d': (0.7, 0.9), 'n': (0.5, 0.7)},
    conditions=["avgWTP_left", "avgWTP_right", "fixation"],
    choice_names=("left", "right"),
    T_dur=30,
    dx=0.01,
    dt=dt
)

model._overlay = pyddm.models.OverlayChain(overlays=[])

model.fit(sample=sample, verbose=True)

Info: Model(name='n', drift=DriftEasy(d=Fitted(0.8282891101916723, minval=0.7, maxval=0.9)), noise=NoiseEasy(n=Fitted(0.6158219082295315, minval=0.5, maxval=0.7)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=591.9436366110376
Info: Model(name='n', drift=DriftEasy(d=Fitted(0.8196291290416825, minval=0.7, maxval=0.9)), noise=NoiseEasy(n=Fitted(0.601097686166267, minval=0.5, maxval=0.7)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=604.2039313056664
Info: Model(name='n', drift=DriftEasy(d=Fitted(0.779006769433045, minval=0.7, maxval=0.9)), noise=NoiseEasy(n=Fitted(0.578887804505553, minval=0.5, maxval=0.7)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=606.3353012793119
Info: Mode

differential_evolution step 1: f(x)= 465.03514218321453


Info: Model(name='n', drift=DriftEasy(d=Fitted(0.7893363634748787, minval=0.7, maxval=0.9)), noise=NoiseEasy(n=Fitted(0.69248366453161, minval=0.5, maxval=0.7)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=498.2999443472073
Info: Model(name='n', drift=DriftEasy(d=Fitted(0.7803783945913365, minval=0.7, maxval=0.9)), noise=NoiseEasy(n=Fitted(0.6998452980919702, minval=0.5, maxval=0.7)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=488.488543538651
Info: Model(name='n', drift=DriftEasy(d=Fitted(0.7147880975777976, minval=0.7, maxval=0.9)), noise=NoiseEasy(n=Fitted(0.6707191884750412, minval=0.5, maxval=0.7)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=476.2180026250724
Info: Mode

differential_evolution step 2: f(x)= 465.03514218321453


Info: Model(name='n', drift=DriftEasy(d=Fitted(0.743555512354034, minval=0.7, maxval=0.9)), noise=NoiseEasy(n=Fitted(0.6755863534980053, minval=0.5, maxval=0.7)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=487.33216714562167
Info: Model(name='n', drift=DriftEasy(d=Fitted(0.7831673724616536, minval=0.7, maxval=0.9)), noise=NoiseEasy(n=Fitted(0.5988069655723081, minval=0.5, maxval=0.7)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=583.7399819139927
Info: Model(name='n', drift=DriftEasy(d=Fitted(0.7461920944062256, minval=0.7, maxval=0.9)), noise=NoiseEasy(n=Fitted(0.6005939912518032, minval=0.5, maxval=0.7)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=558.5958180186711
Info: M

...

Info: Model(name='n', drift=DriftEasy(d=Fitted(0.7007576835088059, minval=0.7, maxval=0.9)), noise=NoiseEasy(n=Fitted(0.6861858611240711, minval=0.5, maxval=0.7)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=458.3646717803056
Info: Model(name='n', drift=DriftEasy(d=Fitted(0.7022021076600476, minval=0.7, maxval=0.9)), noise=NoiseEasy(n=Fitted(0.6504315864645087, minval=0.5, maxval=0.7)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=485.4067308507004
Info: Model(name='n', drift=DriftEasy(d=Fitted(0.7080975375045996, minval=0.7, maxval=0.9)), noise=NoiseEasy(n=Fitted(0.5613558172945248, minval=0.5, maxval=0.7)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=582.8159631493976
Info: M

differential_evolution step 9: f(x)= 450.00147945706004


Info: Model(name='n', drift=DriftEasy(d=Fitted(0.7035828073777017, minval=0.7, maxval=0.9)), noise=NoiseEasy(n=Fitted(0.6990994904588885, minval=0.5, maxval=0.7)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=451.42715352459044
Info: Model(name='n', drift=DriftEasy(d=Fitted(0.7097680727743086, minval=0.7, maxval=0.9)), noise=NoiseEasy(n=Fitted(0.6858822872923053, minval=0.5, maxval=0.7)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=462.98314232316216
Info: Model(name='n', drift=DriftEasy(d=Fitted(0.7088292682312238, minval=0.7, maxval=0.9)), noise=NoiseEasy(n=Fitted(0.644429110243888, minval=0.5, maxval=0.7)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=493.9228229195081
Info: 

differential_evolution step 10: f(x)= 449.9657218830627
Polishing solution with 'L-BFGS-B'


Info: Model(name='n', drift=DriftEasy(d=Fitted(0.701068013324401, minval=0.7, maxval=0.9)), noise=NoiseEasy(n=Fitted(0.6995288245949242, minval=0.5, maxval=0.7)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=449.9657218830627
Info: Model(name='n', drift=DriftEasy(d=Fitted(0.7010680233244011, minval=0.7, maxval=0.9)), noise=NoiseEasy(n=Fitted(0.6995288245949242, minval=0.5, maxval=0.7)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=449.96572663071464
Info: Model(name='n', drift=DriftEasy(d=Fitted(0.701068013324401, minval=0.7, maxval=0.9)), noise=NoiseEasy(n=Fitted(0.6995288345949242, minval=0.5, maxval=0.7)), bound=BoundConstant(B=1), IC=ICPointRatio(x0=0), overlay=OverlayChain(overlays=[]), dx=0.01, dt=0.01, T_dur=30, choice_names=('left', 'right')) loss=449.96571571057507
Info: M